# 1 таск

In [ ]:
# !pip install -q --upgrade transformers==4.57.1 trl==0.24.0 peft==0.17.1 accelerate==1.10.1 bitsandbytes==0.47.0 datasets==3.6.0

In [1]:
import random
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

from transformers import set_seed

import json

import numpy as np
import torch

SEED = 42
DATA_DIR = Path("/kaggle/input/datasets/lambdaderta/red-cu-task/ml-olympiad-red-task")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [2]:
def load_jsonl(name):
    path = DATA_DIR / "data" / name
    with open(path, encoding="utf-8") as file:
        return [json.loads(line) for line in file]

kid_adult = load_jsonl("kid_adult.jsonl")
good_bad = load_jsonl("good_bad.jsonl")
test_style = load_jsonl("public_test_style.jsonl")
test_quality = load_jsonl("public_test_quality.jsonl")

In [3]:
kid_adult[1]

{'prompt': 'Что такое химическая реакция? Это процесс, при котором одни вещества превращаются в совершенно другие.',
 'kid': 'Это когда вещества встречаются и превращаются во что-то совсем новое, как когда из сырых продуктов в духовке получается вкусный пирог. Настоящее волшебство происходит, когда обычное тесто меняется и становится румяным хлебом. Старые вещества исчезают, а на их месте появляются совершенно другие.',
 'adult': 'Химическая реакция — это процесс превращения исходных веществ (реагентов) в новые соединения (продукты) путем разрыва и образования химических связей между атомами. При этом состав атомов в системе сохраняется, но их пространственное строение и свойства кардинально меняются. Процесс всегда сопровождается поглощением или выделением энергии в виде тепла или света.'}

In [4]:
good_bad[0]

{'instruction': 'Как кнопка на клавиатуре печатает букву на экране? Ответ: При нажатии контакты внутри замыкаются, и сигнал передается в мозг компьютера, который рисует нужную букву.',
 'chosen': 'Под кнопкой есть маленькая кнопочка-переключатель. Когда ты её нажимаешь, она посылает компьютеру тайный код из электричества. Компьютер мгновенно разгадывает этот код и рисует нужную букву на экране, прямо как волшебник!',
 'rejected': 'Знаешь, под каждой кнопочкой на твоей клавиатуре спрятана специальная маленькая пружинка и крошечная площадка. Как только ты нажимаешь пальцем на кнопку, она опускается вниз, и эта площадочка соединяет две проводящие штучки вместе. В этот самый момент по проводочку внутри компьютера бежит невидимый электрический сигнал, который очень-очень быстро добегает до самого главного мозга, процессора. Процессор тут же понимает, какую именно букву ты хотел написать, и отправляет команду экрану, чтобы она тут же появилась!'}

In [5]:
test_style[0]

{'prompt': 'Почему продавцы в магазинах делают скидки и распродажи? Ответ: Чтобы быстрее распродать старые товары, освободить место для новых коллекций или привлечь больше покупателей.',
 'kid': 'Магазину нужно освободить полки, чтобы поставить новые игрушки и вещи. Поэтому старые товары продают дешевле, как будто кричат: «Забирайте скорее!». Это здорово помогает быстро всё разобрать и порадовать покупателей.',
 'adult': 'Продавцы делают распродажи для ускорения оборачиваемости товаров и освобождения складских запасов перед закупкой новых коллекций. Маркетинговое снижение цены стимулирует покупательский спрос и позволяет конвертировать неликвидные активы в живые деньги, избегая замораживания капитала.',
 'fact': 'Скидки и распродажи используются для ускорения сбыта остатков, расчистки складов под новые поступления и стимуляции спроса.'}

In [6]:
test_quality[0]

{'prompt': 'Почему у металла и у дерева разная температура в одной комнате? Ответ: Металл очень быстро забирает тепло у нашей руки, поэтому нам он кажется холоднее дерева.',
 'chosen': 'На самом деле они одинаковые! Просто металл — отличный мостик для тепла, поэтому он очень быстро забирает его у нашей руки, и нам кажется, что он холоднее.',
 'rejected': 'Знаешь, это очень-очень интересный вопрос! Представь себе, что они оба совершенно одинаковой температуры, как и сам воздух в твоей комнате. Но дело в том, что когда ты дотрагиваешься до металла ручкой, он просто-напросто начинает быстро-быстро вытягивать всё тепло из твоей руки. Дерево тоже немножко забирает тепло, но оно делает это совсем-совсем медленно, поэтому наша рука не успевает замёрзнуть, и нам кажется, что оно более тёпленькое и приятное на ощупь!'}

In [7]:
# ! pip install -U bitsandbytes

In [8]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [9]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear4bit(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
 

In [10]:
examples = []

for row in kid_adult:
    example = {
        "prompt": [
            {"role": "user", "content": row["prompt"]}
        ],
        "completion": [
            {"role": "assistant", "content": row["kid"]}
        ],
    }

    examples.append(example)

sft_data = Dataset.from_list(examples)

In [11]:
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

In [12]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [13]:
# ! pip install trl

In [14]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="sft_style",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_length=512,
    fp16=True,
    logging_steps=25,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    loss_type="nll",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=sft_data,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

In [15]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as s

Step,Training Loss
25,1.558200
50,1.326600
75,1.261700
100,1.274200
125,1.225700
150,1.210300
175,1.191400
200,1.044700
225,0.880800
250,0.846600


TrainOutput(global_step=561, training_loss=0.9069820504350034, metrics={'train_runtime': 3246.6426, 'train_samples_per_second': 1.376, 'train_steps_per_second': 0.173, 'total_flos': 1.33823652761088e+16, 'train_loss': 0.9069820504350034, 'entropy': 0.9636641542116801, 'num_tokens': 608310.0, 'mean_token_accuracy': 0.8478224564481665, 'epoch': 3.0})

In [16]:
adapter_path = "/kaggle/working/sft_style_adapter"

trainer.save_model(adapter_path)
tokenizer.save_pretrained(adapter_path)

('/kaggle/working/sft_style_adapter/tokenizer_config.json',
 '/kaggle/working/sft_style_adapter/special_tokens_map.json',
 '/kaggle/working/sft_style_adapter/chat_template.jinja',
 '/kaggle/working/sft_style_adapter/vocab.json',
 '/kaggle/working/sft_style_adapter/merges.txt',
 '/kaggle/working/sft_style_adapter/added_tokens.json',
 '/kaggle/working/sft_style_adapter/tokenizer.json')

In [17]:
import pickle

model.config.use_cache = True
model.eval()

with open(DATA_DIR / "metrics" / "style_clf.pkl", "rb") as file:
    style_clf = pickle.load(file)

answers = []

for row in test_style:
    messages = [
        {"role": "user", "content": row["prompt"]}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    start = inputs["input_ids"].shape[1]
    new_tokens = output[0][start:]

    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
    answers.append(answer)

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.6.1. This might lead to breaking c

AttributeError: 'dict' object has no attribute 'predict_proba'

In [18]:
style_clf.keys()

dict_keys(['clf', 'vecs'])

## P_simple подсчет

In [19]:
from scipy.sparse import hstack

vectors = []

for vec in style_clf["vecs"]:
    vector = vec.transform(answers)
    vectors.append(vector)

features = hstack(vectors)

probabilities = style_clf["clf"].predict_proba(features)[:, 1]
p_simple = float(np.mean(probabilities))

print("P_simple:", p_simple)

P_simple: 0.9703538977842824


# 2 таск

In [21]:
model.load_adapter(
    adapter_path,
    adapter_name="reference",
    is_trainable=False,
)

model.set_adapter("default")
model.config.use_cache = False

In [22]:
dpo_examples = []

for row in kid_adult:
    example = {
        "prompt": [
            {"role": "user", "content": row["prompt"]}
        ],
        "chosen": [
            {"role": "assistant", "content": row["kid"]}
        ],
        "rejected": [
            {"role": "assistant", "content": row["adult"]}
        ],
    }

    dpo_examples.append(example)

dpo_data = Dataset.from_list(dpo_examples)

In [23]:
from trl import DPOConfig, DPOTrainer

dpo_args = DPOConfig(
    output_dir="dpo_style",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    beta=0.1,
    max_length=512,
    max_prompt_length=256,
    fp16=True,
    logging_steps=25,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    model_adapter_name="default",
    ref_adapter_name="reference",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dpo_data,
    processing_class=tokenizer,
)

Extracting prompt in train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

In [24]:
dpo_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss
25,0.229100
50,0.010100
75,0.002600
100,0.001700
125,0.002000
150,0.000900
175,0.001400


TrainOutput(global_step=187, training_loss=0.033168169161812824, metrics={'train_runtime': 2242.8887, 'train_samples_per_second': 0.664, 'train_steps_per_second': 0.083, 'total_flos': 0.0, 'train_loss': 0.033168169161812824, 'epoch': 1.0})

In [25]:
dpo_path = "/kaggle/working/dpo_style_adapter"

trainer.save_model(dpo_path)
tokenizer.save_pretrained(dpo_path)

('/kaggle/working/dpo_style_adapter/tokenizer_config.json',
 '/kaggle/working/dpo_style_adapter/special_tokens_map.json',
 '/kaggle/working/dpo_style_adapter/chat_template.jinja',
 '/kaggle/working/dpo_style_adapter/vocab.json',
 '/kaggle/working/dpo_style_adapter/merges.txt',
 '/kaggle/working/dpo_style_adapter/added_tokens.json',
 '/kaggle/working/dpo_style_adapter/tokenizer.json')

In [28]:
import json

public_test_style = []

with open(f"{DATA_DIR}/data/public_test_style.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        row = json.loads(line)
        public_test_style.append(row)

print(len(public_test_style))
print(public_test_style[0])

50
{'prompt': 'Почему продавцы в магазинах делают скидки и распродажи? Ответ: Чтобы быстрее распродать старые товары, освободить место для новых коллекций или привлечь больше покупателей.', 'kid': 'Магазину нужно освободить полки, чтобы поставить новые игрушки и вещи. Поэтому старые товары продают дешевле, как будто кричат: «Забирайте скорее!». Это здорово помогает быстро всё разобрать и порадовать покупателей.', 'adult': 'Продавцы делают распродажи для ускорения оборачиваемости товаров и освобождения складских запасов перед закупкой новых коллекций. Маркетинговое снижение цены стимулирует покупательский спрос и позволяет конвертировать неликвидные активы в живые деньги, избегая замораживания капитала.', 'fact': 'Скидки и распродажи используются для ускорения сбыта остатков, расчистки складов под новые поступления и стимуляции спроса.'}


In [29]:
import torch
from transformers import set_seed
from tqdm.auto import tqdm

set_seed(42)

dpo_model = trainer.model
dpo_model.set_adapter("default")
dpo_model.eval()

answers = []

for row in tqdm(public_test_style):
    messages = [
        {"role": "user", "content": row["prompt"]}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to(dpo_model.device)

    with torch.no_grad():
        output = dpo_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[0, inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    answers.append(answer)

  0%|          | 0/50 [00:00<?, ?it/s]

## Метрика после дпо

In [32]:
import joblib
from scipy.sparse import hstack

style_data = joblib.load("/kaggle/input/datasets/lambdaderta/red-cu-task/ml-olympiad-red-task/metrics/style_clf.pkl")

clf = style_data["clf"]
vecs = style_data["vecs"]

vectors = []

for vec in vecs:
    transformed_answers = vec.transform(answers)
    vectors.append(transformed_answers)

answer_vectors = hstack(vectors)

probabilities = clf.predict_proba(answer_vectors)[:, 1]
p_simple_dpo = float(np.mean(probabilities))

print("DPO style P_simple:", p_simple_dpo)
print(answers[0])

DPO style P_simple: 0.9862012560198689
Продавцы делают скидки, чтобы старые игрушки и вещи быстро продать. Это освободит место для самых новых и классных вещей. А еще так становится проще купить что-то вкусное или красивое, потому что теперь оно стоит дешевле!


# 3 таск

In [34]:
import gc
import torch

trainer = None
model = None
dpo_model = None
answers = None

gc.collect()
torch.cuda.empty_cache()

In [36]:
good_bad = []

with open(f"{DATA_DIR}/data/good_bad.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        row = json.loads(line)
        good_bad.append(row)

len(good_bad)

2226

In [37]:
from datasets import Dataset

rm_rows = []

for row in good_bad:
    chosen_messages = [
        {
            "role": "user",
            "content": row["instruction"],
        },
        {
            "role": "assistant",
            "content": row["chosen"],
        },
    ]

    rejected_messages = [
        {
            "role": "user",
            "content": row["instruction"],
        },
        {
            "role": "assistant",
            "content": row["rejected"],
        },
    ]

    new_row = {
        "chosen": chosen_messages,
        "rejected": rejected_messages,
    }

    rm_rows.append(new_row)

rm_data = Dataset.from_list(rm_rows)

In [38]:
rm_data = rm_data.train_test_split(
    test_size=0.1,
    seed=42,
)

rm_train = rm_data["train"]
rm_val = rm_data["test"]

rm_train[0]

{'chosen': [{'role': 'user',
   'content': 'Почему одеяло сохраняет нас в тепле? Ткань не дает телу нагреться, она просто ловит тепло, которое вырабатывает наше тело, и не пускает холодный воздух.'},
  {'role': 'assistant',
   'content': 'Ты абсолютно прав, одеяло работает как тёплый домик, который сам не греет, а бережёт твоё тепло! Между ниточками ткани прячется воздух, который не пускает тепло нашего тела наружу.'}],
 'rejected': [{'role': 'user',
   'content': 'Почему одеяло сохраняет нас в тепле? Ткань не дает телу нагреться, она просто ловит тепло, которое вырабатывает наше тело, и не пускает холодный воздух.'},
  {'role': 'assistant',
   'content': 'Ой, ты такой умный, что сам всё понял, но давай я расскажу подробнее! Знаешь, одеяло вообще-то не живое и в нём нет батарейки, поэтому оно совсем не умеет греть, как солнышко. Просто представь, что одеяло   это такая большая и пушистая стена, которая накрыла тебя сверху. И вот ты лежишь под ней и вырабатываешь тепло, а оно берёт и не

In [39]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
)

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

rm_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

rm_model.config.pad_token_id = tokenizer.pad_token_id
rm_model.config.problem_type = "regression"
rm_model.config.use_cache = False

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-4B-Instruct-2507 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [40]:
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

rm_model = prepare_model_for_kbit_training(rm_model)

rm_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    modules_to_save=["score"],
)

rm_model = get_peft_model(
    rm_model,
    rm_lora_config,
)

rm_model.print_trainable_parameters()

trainable params: 33,032,704 || all params: 4,055,503,360 || trainable%: 0.8145


In [42]:
from trl import RewardConfig, RewardTrainer

rm_args = RewardConfig(
    output_dir="/kaggle/working/rm_checkpoints",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    max_length=512,

    fp16=True,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },

    optim="paged_adamw_8bit",

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,

    logging_steps=25,
    save_total_limit=2,

    seed=42,
    data_seed=42,
    report_to="none",
)

In [43]:
rm_trainer = RewardTrainer(
    model=rm_model,
    args=rm_args,
    train_dataset=rm_train,
    eval_dataset=rm_val,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/2003 [00:00<?, ? examples/s]

Filtering train >512 tokens:   0%|          | 0/2003 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/223 [00:00<?, ? examples/s]

Filtering eval >512 tokens:   0%|          | 0/223 [00:00<?, ? examples/s]

In [44]:
rm_trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Num Tokens,Min Reward,Mean Reward,Max Reward,Accuracy,Margin
1,0.043700,0.143944,597387.000000,-18.019733,2.726619,23.472971,0.991031,41.210613


TrainOutput(global_step=251, training_loss=0.05498169923785135, metrics={'train_runtime': 2264.457, 'train_samples_per_second': 0.885, 'train_steps_per_second': 0.111, 'total_flos': 1.47702747419136e+16, 'train_loss': 0.05498169923785135, 'epoch': 1.0})

In [45]:
rm_path = "/kaggle/working/reward_model_adapter"

rm_trainer.save_model(rm_path)
tokenizer.save_pretrained(rm_path)

('/kaggle/working/reward_model_adapter/tokenizer_config.json',
 '/kaggle/working/reward_model_adapter/special_tokens_map.json',
 '/kaggle/working/reward_model_adapter/chat_template.jinja',
 '/kaggle/working/reward_model_adapter/vocab.json',
 '/kaggle/working/reward_model_adapter/merges.txt',
 '/kaggle/working/reward_model_adapter/added_tokens.json',
 '/kaggle/working/reward_model_adapter/tokenizer.json')

In [46]:
import json

public_test_quality = []

with open(
    f"{DATA_DIR}/data/public_test_quality.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        row = json.loads(line)
        public_test_quality.append(row)

len(public_test_quality)

50

In [47]:
import torch

rm_model = rm_trainer.model
rm_model.eval()

def get_reward(instruction, answer):
    messages = [
        {
            "role": "user",
            "content": instruction,
        },
        {
            "role": "assistant",
            "content": answer,
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=False,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
        truncation=True,
        max_length=512,
    ).to(rm_model.device)

    with torch.no_grad():
        output = rm_model(**inputs)

    score = output.logits[0, 0].float().item()

    return score

## метрика 3 таск

In [49]:
correct = 0
margins = []

for row in tqdm(public_test_quality):
    chosen_score = get_reward(
        row["prompt"],
        row["chosen"],
    )

    rejected_score = get_reward(
        row["prompt"],
        row["rejected"],
    )

    if chosen_score > rejected_score:
        correct += 1

    margins.append(chosen_score - rejected_score)

rm_accuracy = correct / len(public_test_quality)
mean_margin = float(np.mean(margins))

print("Public RM accuracy:", rm_accuracy)
print("Public mean margin:", mean_margin)

  0%|          | 0/50 [00:00<?, ?it/s]

Public RM accuracy: 1.0
Public mean margin: 45.133200073242186


# 4 таска да что так много черный лвл быстрее решается

In [50]:
rm_trainer = None
rm_model = None

gc.collect()
torch.cuda.empty_cache()

In [51]:
quality_rows = []

for row in good_bad:
    new_row = {
        "prompt": [
            {
                "role": "user",
                "content": row["instruction"],
            }
        ],
        "chosen": [
            {
                "role": "assistant",
                "content": row["chosen"],
            }
        ],
        "rejected": [
            {
                "role": "assistant",
                "content": row["rejected"],
            }
        ],
    }

    quality_rows.append(new_row)

quality_data = Dataset.from_list(quality_rows)

In [52]:
quality_data = quality_data.train_test_split(
    test_size=0.1,
    seed=42,
)

quality_train = quality_data["train"]
quality_val = quality_data["test"]

quality_train[0]

{'prompt': [{'role': 'user',
   'content': 'Почему одеяло сохраняет нас в тепле? Ткань не дает телу нагреться, она просто ловит тепло, которое вырабатывает наше тело, и не пускает холодный воздух.'}],
 'chosen': [{'role': 'assistant',
   'content': 'Ты абсолютно прав, одеяло работает как тёплый домик, который сам не греет, а бережёт твоё тепло! Между ниточками ткани прячется воздух, который не пускает тепло нашего тела наружу.'}],
 'rejected': [{'role': 'assistant',
   'content': 'Ой, ты такой умный, что сам всё понял, но давай я расскажу подробнее! Знаешь, одеяло вообще-то не живое и в нём нет батарейки, поэтому оно совсем не умеет греть, как солнышко. Просто представь, что одеяло   это такая большая и пушистая стена, которая накрыла тебя сверху. И вот ты лежишь под ней и вырабатываешь тепло, а оно берёт и не пускает его на волю гулять.'}]}

In [53]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
DPO_STYLE_PATH = "/kaggle/working/dpo_style_adapter"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

base_model = prepare_model_for_kbit_training(
    base_model,
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [55]:
from peft import (
    PeftModel,
    prepare_model_for_kbit_training,
)

quality_model = PeftModel.from_pretrained(
    base_model,
    DPO_STYLE_PATH,
    adapter_name="default",
    is_trainable=True,
)

In [56]:
quality_model.load_adapter(
    DPO_STYLE_PATH,
    adapter_name="reference",
    is_trainable=False,
)

quality_model.set_adapter("default")
quality_model.config.use_cache = False

quality_model.print_trainable_parameters()

trainable params: 33,030,144 || all params: 4,088,528,384 || trainable%: 0.8079


In [57]:
tokenizer.padding_side = "left"

In [58]:
from trl import DPOConfig, DPOTrainer

quality_args = DPOConfig(
    output_dir="/kaggle/working/dpo_quality_checkpoints",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=5e-6,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    beta=0.1,

    max_length=512,
    max_prompt_length=256,

    fp16=True,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },

    optim="paged_adamw_8bit",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    logging_steps=25,

    model_adapter_name="default",
    ref_adapter_name="reference",

    seed=42,
    data_seed=42,
    report_to="none",
)

In [59]:
quality_trainer = DPOTrainer(
    model=quality_model,
    ref_model=None,

    args=quality_args,

    train_dataset=quality_train,
    eval_dataset=quality_val,

    processing_class=tokenizer,
)

Extracting prompt in train dataset:   0%|          | 0/2003 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/2003 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2003 [00:00<?, ? examples/s]

Extracting prompt in eval dataset:   0%|          | 0/223 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/223 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/223 [00:00<?, ? examples/s]

In [60]:
quality_trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
1,0.033600,0.039874,-1.324770,-7.598870,0.991031,6.274101,-149.708725,-336.550659,-2.670070,-2.981051


TrainOutput(global_step=251, training_loss=0.1209559385531185, metrics={'train_runtime': 3310.054, 'train_samples_per_second': 0.605, 'train_steps_per_second': 0.076, 'total_flos': 0.0, 'train_loss': 0.1209559385531185, 'epoch': 1.0})

In [61]:
quality_path = "/kaggle/working/dpo_quality_adapter"

quality_trainer.save_model(quality_path)
tokenizer.save_pretrained(quality_path)

('/kaggle/working/dpo_quality_adapter/tokenizer_config.json',
 '/kaggle/working/dpo_quality_adapter/special_tokens_map.json',
 '/kaggle/working/dpo_quality_adapter/chat_template.jinja',
 '/kaggle/working/dpo_quality_adapter/vocab.json',
 '/kaggle/working/dpo_quality_adapter/merges.txt',
 '/kaggle/working/dpo_quality_adapter/added_tokens.json',
 '/kaggle/working/dpo_quality_adapter/tokenizer.json')

In [62]:
import torch.nn.functional as F

quality_model = quality_trainer.model
quality_model.set_adapter("default")
quality_model.eval()

def mean_answer_logprob(prompt, answer):
    prompt_messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        add_generation_prompt=True,
        tokenize=True,
    )

    answer_ids = tokenizer(
        answer,
        add_special_tokens=False,
    )["input_ids"]

    all_ids = prompt_ids + answer_ids

    input_ids = torch.tensor(
        [all_ids],
        device=quality_model.device,
    )

    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        output = quality_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

    logits = output.logits[0]

    answer_logits = logits[
        len(prompt_ids) - 1 : len(all_ids) - 1
    ]

    answer_targets = input_ids[
        0,
        len(prompt_ids) :
    ]

    log_probs = F.log_softmax(
        answer_logits.float(),
        dim=-1,
    )

    answer_log_probs = log_probs.gather(
        dim=1,
        index=answer_targets.unsqueeze(1),
    ).squeeze(1)

    return answer_log_probs.mean().item()

# метрика 4 таск

In [63]:
correct = 0
margins = []

for row in tqdm(public_test_quality):
    chosen_logprob = mean_answer_logprob(
        row["prompt"],
        row["chosen"],
    )

    rejected_logprob = mean_answer_logprob(
        row["prompt"],
        row["rejected"],
    )

    if chosen_logprob > rejected_logprob:
        correct += 1

    margins.append(
        chosen_logprob - rejected_logprob
    )

quality_accuracy = correct / len(public_test_quality)
quality_mean_margin = float(np.mean(margins))

print("DPO quality accuracy:", quality_accuracy)
print("DPO quality mean margin:", quality_mean_margin)

  0%|          | 0/50 [00:00<?, ?it/s]

DPO quality accuracy: 1.0
DPO quality mean margin: 1.5127040123939515
